# Project 3 Milestone 4 — Distance / Parallax Recovery for Angular Momentum Analysis

This notebook addresses the Project 3 Milestone 3 readiness issue: the orbital-diagnostic candidate table preserved velocity-space information, but the 27 candidates did not yet carry explicit parallax or distance columns needed for angular-momentum calculations.

Goal:
- inspect existing processed Gaia–LAMOST larger feature / candidate tables;
- recover Gaia parallax where available by merging on stable identifiers;
- derive a simple exploratory inverse-parallax distance for positive parallaxes;
- create a distance-recovered Project 3 candidate table;
- document readiness for later `Lz`, `Lperp`, and `Ltot` calculations.

Important note:
The inverse-parallax distance here is a practical readiness step, not a final astrophysical distance model. For publication-grade orbit work, this should later be replaced or validated with a more careful distance estimate and parallax zero-point treatment.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Resolve repository root robustly.
# When executed through nbconvert, cwd may be notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not resolve repository root containing data/processed")

DATA = ROOT / "data" / "processed"
REPORT = ROOT / "report"

candidate_path = DATA / "project3_orbital_diagnostics_candidates.csv"

candidate_path.exists(), candidate_path

(True,
 PosixPath('/Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/project3_orbital_diagnostics_candidates.csv'))

In [2]:
candidates = pd.read_csv(candidate_path)

print("Project 3 M3 candidates:", candidates.shape)
display(candidates.head())
print("\nColumns:")
print(list(candidates.columns))

Project 3 M3 candidates: (27, 49)


,source_id,ra,dec,parallax,pmra,pmdec,radial_velocity,bp_rp,absolute_g_mag,feh,tangential_velocity_kms,galcen_vx,galcen_vy,galcen_vz,galcen_vtot,has_astrometric_6d_input,has_galcen_velocity_input,orbit_input_ready,has_positive_parallax,has_distance_or_parallax_input,can_compute_galcen_position,can_compute_angular_momentum,missing_orbital_position_input,galcen_vx_kms,galcen_vy_kms,galcen_vz_kms,galcen_vtot_recomputed_kms,galcen_vtot_delta_kms,distance_kpc_used,galcen_x_kpc,galcen_y_kpc,galcen_z_kpc,galcen_radius_kpc,galcen_cyl_radius_kpc,galcen_abs_z_kpc,Lx_kpc_kms,Ly_kpc_kms,Lz_kpc_kms,Lperp_kpc_kms,Ltot_kpc_kms,Lperp_over_Ltot,Lz_over_Ltot,abs_Lz_over_Ltot,retrograde_flag,disk_like_orbit_flag,halo_like_orbit_flag,high_galcen_velocity_flag,high_tangential_velocity_flag,metal_poor_flag
0,3077457042404665088,NaN,NaN,NaN,NaN,NaN,NaN,1.252923,1.686704,-0.599,111.015798,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False
1,3089847099636770560,NaN,NaN,NaN,NaN,NaN,NaN,0.618819,3.726077,-2.213,413.706467,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True
2,3089944573918027648,NaN,NaN,NaN,NaN,NaN,NaN,0.984529,1.568062,-1.472,426.686109,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True
3,3089534353001157632,NaN,NaN,NaN,NaN,NaN,NaN,1.041462,0.871324,-1.539,376.116952,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True
4,3084095863550902528,NaN,NaN,NaN,NaN,NaN,NaN,0.728496,2.774414,-1.723,357.805943,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True



Columns:
['source_id', 'ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity', 'bp_rp', 'absolute_g_mag', 'feh', 'tangential_velocity_kms', 'galcen_vx', 'galcen_vy', 'galcen_vz', 'galcen_vtot', 'has_astrometric_6d_input', 'has_galcen_velocity_input', 'orbit_input_ready', 'has_positive_parallax', 'has_distance_or_parallax_input', 'can_compute_galcen_position', 'can_compute_angular_momentum', 'missing_orbital_position_input', 'galcen_vx_kms', 'galcen_vy_kms', 'galcen_vz_kms', 'galcen_vtot_recomputed_kms', 'galcen_vtot_delta_kms', 'distance_kpc_used', 'galcen_x_kpc', 'galcen_y_kpc', 'galcen_z_kpc', 'galcen_radius_kpc', 'galcen_cyl_radius_kpc', 'galcen_abs_z_kpc', 'Lx_kpc_kms', 'Ly_kpc_kms', 'Lz_kpc_kms', 'Lperp_kpc_kms', 'Ltot_kpc_kms', 'Lperp_over_Ltot', 'Lz_over_Ltot', 'abs_Lz_over_Ltot', 'retrograde_flag', 'disk_like_orbit_flag', 'halo_like_orbit_flag', 'high_galcen_velocity_flag', 'high_tangential_velocity_flag', 'metal_poor_flag']


In [3]:
# Candidate source tables likely to contain Gaia astrometric fields.
source_paths = [
    DATA / "gaia_lamost_larger_velocity_features.csv",
    DATA / "gaia_lamost_larger_chemo_kinematic_candidates.csv",
    DATA / "project2_candidate_cross_method_summary.csv",
    DATA / "project2_pca_embedding.csv",
    DATA / "project3_orbit_input_candidates.csv",
]

available_sources = [p for p in source_paths if p.exists()]
print("Available source tables:")
for p in available_sources:
    print("-", p, pd.read_csv(p, nrows=1).shape)

if not available_sources:
    raise FileNotFoundError("No candidate source tables found for parallax/distance recovery.")

Available source tables:
- /Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/gaia_lamost_larger_velocity_features.csv (1, 47)
- /Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/gaia_lamost_larger_chemo_kinematic_candidates.csv (1, 31)
- /Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/project2_candidate_cross_method_summary.csv (1, 24)
- /Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/project3_orbit_input_candidates.csv (1, 18)


In [4]:
def summarize_columns(path):
    df0 = pd.read_csv(path, nrows=5)
    cols = list(df0.columns)
    interesting = [
        c for c in cols
        if any(k in c.lower() for k in [
            "source", "ra", "dec", "parallax", "distance", "pmra", "pmdec", "rv",
            "radial", "feh", "vtan", "galcen"
        ])
    ]
    return cols, interesting

for p in available_sources:
    cols, interesting = summarize_columns(p)
    print("\n==", p.name, "==")
    print("n_columns:", len(cols))
    print("interesting columns:")
    print(interesting)


== gaia_lamost_larger_velocity_features.csv ==
n_columns: 47
interesting columns:
['source_id', 'ra_gaia', 'dec_gaia', 'parallax', 'parallax_over_error', 'pmra', 'pmdec', 'distance_pc', 'ra_lamost', 'dec_lamost', 'feh', 'rv', 'high_vtan_candidate', 'galcen_x_kpc', 'galcen_y_kpc', 'galcen_z_kpc', 'galcen_vx_kms', 'galcen_vy_kms', 'galcen_vz_kms', 'galcen_vtot_kms']

== gaia_lamost_larger_chemo_kinematic_candidates.csv ==
n_columns: 31
interesting columns:
['source_id', 'ra_gaia', 'dec_gaia', 'parallax', 'distance_pc', 'feh', 'rv', 'galcen_x_kpc', 'galcen_y_kpc', 'galcen_z_kpc', 'galcen_vx_kms', 'galcen_vy_kms', 'galcen_vz_kms', 'galcen_vtot_kms', 'high_vtan_candidate']

== project2_candidate_cross_method_summary.csv ==
n_columns: 24
interesting columns:
['source_id', 'feh', 'galcen_vtot_kms', 'high_vtan_candidate', 'pca_noise_stability_fraction', 'umap_noise_stability_fraction']

== project3_orbit_input_candidates.csv ==
n_columns: 18
interesting columns:
['source_id', 'ra', 'dec', 'pa

In [5]:
# Pick a robust merge key.
# Priority: Gaia source_id if available; otherwise fall back to exact coordinate/chemokinematic keys.
def common_cols(a, b):
    return [c for c in a.columns if c in b.columns]

candidate_cols = set(candidates.columns)

preferred_keys = [
    ["source_id"],
    ["gaia_source_id"],
    ["source_id_1"],
    ["ra", "dec"],
    ["ra_1", "dec_1"],
]

def find_key(left, right):
    for key in preferred_keys:
        if all(k in left.columns and k in right.columns for k in key):
            return key
    return []

source_dfs = {}
for p in available_sources:
    df = pd.read_csv(p)
    source_dfs[p.name] = df
    print(p.name, df.shape, "merge_key:", find_key(candidates, df))

gaia_lamost_larger_velocity_features.csv (1838, 47) merge_key: ['source_id']
gaia_lamost_larger_chemo_kinematic_candidates.csv (27, 31) merge_key: ['source_id']
project2_candidate_cross_method_summary.csv (27, 24) merge_key: ['source_id']
project3_orbit_input_candidates.csv (27, 18) merge_key: ['source_id']


In [6]:
# Build a recovery source table from the first source table that has both a valid merge key
# and parallax/distance-like columns.
distance_keywords = ["parallax", "distance", "dist"]
astrometry_keywords = ["ra", "dec", "pmra", "pmdec"]

recovery_attempts = []

for name, df in source_dfs.items():
    key = find_key(candidates, df)
    recover_cols = [
        c for c in df.columns
        if c in key
        or any(k in c.lower() for k in distance_keywords + astrometry_keywords)
        or c.lower() in ["rv", "radial_velocity", "radial_velocity_kms"]
    ]
    has_distance_signal = any(any(k in c.lower() for k in distance_keywords) for c in recover_cols)
    recovery_attempts.append({
        "source": name,
        "shape": df.shape,
        "key": key,
        "recover_cols": recover_cols,
        "has_distance_signal": has_distance_signal,
    })

for item in recovery_attempts:
    print("\nSOURCE:", item["source"])
    print("key:", item["key"])
    print("has_distance_signal:", item["has_distance_signal"])
    print("recover_cols:", item["recover_cols"])


SOURCE: gaia_lamost_larger_velocity_features.csv
key: ['source_id']
has_distance_signal: True
recover_cols: ['source_id', 'ra_gaia', 'dec_gaia', 'parallax', 'parallax_over_error', 'pmra', 'pmdec', 'distance_pc', 'ra_lamost', 'dec_lamost', 'rv']

SOURCE: gaia_lamost_larger_chemo_kinematic_candidates.csv
key: ['source_id']
has_distance_signal: True
recover_cols: ['source_id', 'ra_gaia', 'dec_gaia', 'parallax', 'distance_pc', 'rv']

SOURCE: project2_candidate_cross_method_summary.csv
key: ['source_id']
has_distance_signal: False
recover_cols: ['source_id', 'pca_noise_stability_fraction', 'umap_noise_stability_fraction']

SOURCE: project3_orbit_input_candidates.csv
key: ['source_id']
has_distance_signal: True
recover_cols: ['source_id', 'ra', 'dec', 'parallax', 'pmra', 'pmdec', 'radial_velocity']


In [7]:
# Choose best source:
# 1. must have merge key;
# 2. prefer one with parallax/distance columns;
# 3. prefer larger velocity feature table because it is closest to Gaia-derived feature construction.
ranked = []

for item in recovery_attempts:
    score = 0
    if item["key"]:
        score += 10
    if item["has_distance_signal"]:
        score += 20
    if item["source"] == "gaia_lamost_larger_velocity_features.csv":
        score += 5
    if item["source"] == "gaia_lamost_larger_chemo_kinematic_candidates.csv":
        score += 3
    ranked.append((score, item))

ranked = sorted(ranked, key=lambda x: x[0], reverse=True)

best_score, best = ranked[0]
print("Best recovery source:", best["source"])
print("Score:", best_score)
print("Merge key:", best["key"])
print("Recover columns:", best["recover_cols"])

if best_score < 10:
    raise RuntimeError("Could not identify a reliable merge key for distance/parallax recovery.")

Best recovery source: gaia_lamost_larger_velocity_features.csv
Score: 35
Merge key: ['source_id']
Recover columns: ['source_id', 'ra_gaia', 'dec_gaia', 'parallax', 'parallax_over_error', 'pmra', 'pmdec', 'distance_pc', 'ra_lamost', 'dec_lamost', 'rv']


In [8]:
source = source_dfs[best["source"]].copy()
key = best["key"]

# Avoid duplicate column collisions except for merge keys.
recover_cols = best["recover_cols"]
recover_source = source[recover_cols].drop_duplicates(subset=key)

print("Recovery source subset:", recover_source.shape)
display(recover_source.head())

Recovery source subset: (1838, 11)


,source_id,ra_gaia,dec_gaia,parallax,parallax_over_error,pmra,pmdec,distance_pc,ra_lamost,dec_lamost,rv
0,3089803084812330368,123.585870,1.280288,1.355037,82.269290,-0.344946,0.219658,737.987179,123.585870,1.280288,22.46
1,3090685683412224256,121.883921,1.744536,0.136420,7.009696,-1.741048,0.012936,7330.315591,121.883922,1.744536,96.55
2,3083676090627788288,121.847440,0.277612,0.523568,35.365410,-2.656724,1.883511,1909.970076,121.847440,0.277612,69.04
3,3083712438932111488,122.455011,0.252453,0.673550,27.477602,-1.330628,-1.522767,1484.670000,122.455010,0.252453,23.39
4,3084883801071606528,120.988708,1.818766,0.578844,21.939972,-1.409557,1.102854,1727.580172,120.988708,1.818767,80.68


In [9]:
merged = candidates.merge(
    recover_source,
    on=key,
    how="left",
    suffixes=("", "_recovered")
)

print("Merged candidate table:", merged.shape)
print("Original candidates:", candidates.shape)
print("Recovered rows matched on key:", merged[key].notna().all(axis=1).sum())
display(merged.head())

Merged candidate table: (27, 59)
Original candidates: (27, 49)
Recovered rows matched on key: 27


,source_id,ra,dec,parallax,pmra,pmdec,radial_velocity,bp_rp,absolute_g_mag,feh,tangential_velocity_kms,galcen_vx,galcen_vy,galcen_vz,galcen_vtot,has_astrometric_6d_input,has_galcen_velocity_input,orbit_input_ready,has_positive_parallax,has_distance_or_parallax_input,can_compute_galcen_position,can_compute_angular_momentum,missing_orbital_position_input,galcen_vx_kms,galcen_vy_kms,galcen_vz_kms,galcen_vtot_recomputed_kms,galcen_vtot_delta_kms,distance_kpc_used,galcen_x_kpc,galcen_y_kpc,galcen_z_kpc,galcen_radius_kpc,galcen_cyl_radius_kpc,galcen_abs_z_kpc,Lx_kpc_kms,Ly_kpc_kms,Lz_kpc_kms,Lperp_kpc_kms,Ltot_kpc_kms,Lperp_over_Ltot,Lz_over_Ltot,abs_Lz_over_Ltot,retrograde_flag,disk_like_orbit_flag,halo_like_orbit_flag,high_galcen_velocity_flag,high_tangential_velocity_flag,metal_poor_flag,ra_gaia,dec_gaia,parallax_recovered,parallax_over_error,pmra_recovered,pmdec_recovered,distance_pc,ra_lamost,dec_lamost,rv
0,3077457042404665088,NaN,NaN,NaN,NaN,NaN,NaN,1.252923,1.686704,-0.599,111.015798,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,124.061847,0.009938,0.574711,23.368060,-13.446595,0.577651,1740.005754,124.061913,0.009931,270.86
1,3089847099636770560,NaN,NaN,NaN,NaN,NaN,NaN,0.618819,3.726077,-2.213,413.706467,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True,122.508355,1.280094,0.637151,26.753075,-42.072760,-36.356421,1569.487021,122.508540,1.280247,-11.29
2,3089944573918027648,NaN,NaN,NaN,NaN,NaN,NaN,0.984529,1.568062,-1.472,426.686109,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True,123.954966,1.686693,0.602822,32.416683,-18.772986,-50.908489,1658.864371,123.955020,1.686775,385.54
3,3089534353001157632,NaN,NaN,NaN,NaN,NaN,NaN,1.041462,0.871324,-1.539,376.116952,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True,124.176994,0.737449,0.458766,25.166069,2.906364,-36.283028,2179.762224,124.176981,0.737614,-43.26
4,3084095863550902528,NaN,NaN,NaN,NaN,NaN,NaN,0.728496,2.774414,-1.723,357.805943,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,True,120.430550,0.240808,0.337022,11.091496,11.675831,-22.600204,2967.169602,120.430517,0.240840,9.04


In [10]:
# Normalize parallax and distance columns into explicit Project 3 M4 names.

def first_existing(df, options):
    for c in options:
        if c in df.columns:
            return c
    return None

parallax_col = first_existing(merged, [
    "parallax", "parallax_recovered", "gaia_parallax", "parallax_mas"
])

parallax_error_col = first_existing(merged, [
    "parallax_error", "parallax_error_recovered", "gaia_parallax_error", "parallax_error_mas"
])

distance_col = first_existing(merged, [
    "distance_pc", "distance", "distance_recovered", "dist", "dist_pc", "r_med_geo", "r_med_photogeo"
])

print("Detected parallax column:", parallax_col)
print("Detected parallax error column:", parallax_error_col)
print("Detected distance column:", distance_col)

Detected parallax column: parallax
Detected parallax error column: None
Detected distance column: distance_pc


In [11]:
recovered = merged.copy()

if parallax_col is not None:
    recovered["parallax_mas_recovered"] = pd.to_numeric(recovered[parallax_col], errors="coerce")
else:
    recovered["parallax_mas_recovered"] = np.nan

if parallax_error_col is not None:
    recovered["parallax_error_mas_recovered"] = pd.to_numeric(recovered[parallax_error_col], errors="coerce")
else:
    recovered["parallax_error_mas_recovered"] = np.nan

if distance_col is not None:
    recovered["distance_existing_recovered"] = pd.to_numeric(recovered[distance_col], errors="coerce")
else:
    recovered["distance_existing_recovered"] = np.nan

# Direct inverse parallax estimate:
# parallax in mas -> distance in pc = 1000 / parallax_mas.
positive_parallax = recovered["parallax_mas_recovered"] > 0
recovered["distance_pc_from_parallax"] = np.where(
    positive_parallax,
    1000.0 / recovered["parallax_mas_recovered"],
    np.nan,
)
recovered["distance_kpc_from_parallax"] = recovered["distance_pc_from_parallax"] / 1000.0

recovered["parallax_snr_recovered"] = np.where(
    recovered["parallax_error_mas_recovered"] > 0,
    recovered["parallax_mas_recovered"] / recovered["parallax_error_mas_recovered"],
    np.nan,
)

recovered["distance_recovery_source"] = best["source"]

recovered["distance_quality_flag"] = np.select(
    [
        recovered["parallax_mas_recovered"].isna(),
        recovered["parallax_mas_recovered"] <= 0,
        recovered["parallax_snr_recovered"] >= 5,
        recovered["parallax_snr_recovered"] >= 3,
        recovered["parallax_snr_recovered"] > 0,
    ],
    [
        "missing_parallax",
        "non_positive_parallax",
        "positive_parallax_snr_ge_5",
        "positive_parallax_snr_3_to_5",
        "positive_parallax_low_snr",
    ],
    default="positive_parallax_unknown_snr",
)

display(recovered[[
    *key,
    "parallax_mas_recovered",
    "parallax_error_mas_recovered",
    "parallax_snr_recovered",
    "distance_pc_from_parallax",
    "distance_kpc_from_parallax",
    "distance_quality_flag",
    "distance_recovery_source",
]].head(10))

,source_id,parallax_mas_recovered,parallax_error_mas_recovered,parallax_snr_recovered,distance_pc_from_parallax,distance_kpc_from_parallax,distance_quality_flag,distance_recovery_source
0,3077457042404665088,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
1,3089847099636770560,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
2,3089944573918027648,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
3,3089534353001157632,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
4,3084095863550902528,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
5,3083736976083264768,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
6,3089512263986192512,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
7,3089611597989508224,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
8,3084599680395408128,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv
9,3084590884302328704,NaN,NaN,NaN,NaN,NaN,missing_parallax,gaia_lamost_larger_velocity_features.csv


In [12]:
# Angular momentum readiness check.
# Minimal ingredients for 6D phase-space style angular momentum preparation:
# sky position + proper motions + radial velocity + distance proxy.
# Exact downstream calculation should use Astropy coordinates and a clearly documented Galactic frame.

def has_any_column(df, options):
    return any(c in df.columns for c in options)

readiness_inputs = {
    "ra": has_any_column(recovered, ["ra", "ra_1", "ra_recovered"]),
    "dec": has_any_column(recovered, ["dec", "dec_1", "dec_recovered"]),
    "pmra": has_any_column(recovered, ["pmra", "pmra_recovered"]),
    "pmdec": has_any_column(recovered, ["pmdec", "pmdec_recovered"]),
    "radial_velocity": has_any_column(recovered, ["rv", "radial_velocity", "radial_velocity_kms", "rv_recovered"]),
    "positive_distance_from_parallax": recovered["distance_pc_from_parallax"].notna().any(),
}

for k, v in readiness_inputs.items():
    print(f"{k}: {v}")

required_row_cols = []
for options in [
    ["ra", "ra_1", "ra_recovered"],
    ["dec", "dec_1", "dec_recovered"],
    ["pmra", "pmra_recovered"],
    ["pmdec", "pmdec_recovered"],
    ["rv", "radial_velocity", "radial_velocity_kms", "rv_recovered"],
]:
    col = first_existing(recovered, options)
    if col is not None:
        required_row_cols.append(col)

required_row_cols.append("distance_pc_from_parallax")

recovered["angular_momentum_ready_basic"] = recovered[required_row_cols].notna().all(axis=1)

summary = {
    "n_candidates": len(recovered),
    "n_with_recovered_parallax": int(recovered["parallax_mas_recovered"].notna().sum()),
    "n_with_positive_parallax": int((recovered["parallax_mas_recovered"] > 0).sum()),
    "n_with_inverse_parallax_distance": int(recovered["distance_pc_from_parallax"].notna().sum()),
    "n_basic_angular_momentum_ready": int(recovered["angular_momentum_ready_basic"].sum()),
    "recovery_source": best["source"],
    "merge_key": ", ".join(key),
}

summary

ra: True
dec: True
pmra: True
pmdec: True
radial_velocity: True
positive_distance_from_parallax: False


{'n_candidates': 27,
 'n_with_recovered_parallax': 0,
 'n_with_positive_parallax': 0,
 'n_with_inverse_parallax_distance': 0,
 'n_basic_angular_momentum_ready': 0,
 'recovery_source': 'gaia_lamost_larger_velocity_features.csv',
 'merge_key': 'source_id'}

In [13]:
quality_summary = (
    recovered["distance_quality_flag"]
    .value_counts(dropna=False)
    .rename_axis("distance_quality_flag")
    .reset_index(name="n_candidates")
)

display(quality_summary)

distance_stats = recovered["distance_kpc_from_parallax"].describe()
distance_stats

,distance_quality_flag,n_candidates
0,missing_parallax,27


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: distance_kpc_from_parallax, dtype: float64

In [14]:
output_path = DATA / "project3_distance_recovered_candidates.csv"
summary_path = DATA / "project3_distance_recovery_summary.csv"

recovered.to_csv(output_path, index=False)

summary_df = pd.DataFrame([summary])
summary_df.to_csv(summary_path, index=False)

print("Wrote:", output_path)
print("Wrote:", summary_path)
print("\nSummary:")
display(summary_df)
display(quality_summary)

Wrote: /Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/project3_distance_recovered_candidates.csv
Wrote: /Users/liors/Documents/gaia-lamost-galactic-archaeology/data/processed/project3_distance_recovery_summary.csv

Summary:


,n_candidates,n_with_recovered_parallax,n_with_positive_parallax,n_with_inverse_parallax_distance,n_basic_angular_momentum_ready,recovery_source,merge_key
0,27,0,0,0,0,gaia_lamost_larger_velocity_features.csv,source_id


,distance_quality_flag,n_candidates
0,missing_parallax,27


In [15]:
REPORT.mkdir(exist_ok=True)

report_path = REPORT / "project3_milestone4_distance_parallax_recovery.md"

quality_lines = "\n".join(
    f"- {row.distance_quality_flag}: {row.n_candidates}"
    for row in quality_summary.itertuples(index=False)
)

report_text = f'''# Project 3 Milestone 4 — Distance / Parallax Recovery for Angular Momentum Analysis

## Objective

Project 3 Milestone 3 identified that the readiness-aware orbital diagnostics candidate table preserved useful velocity-space diagnostics but did not yet carry explicit parallax or distance information for the 27 Project 3 candidates.

This milestone checks whether the missing distance/parallax information can be recovered from existing processed Gaia–LAMOST larger feature or candidate tables, then writes a distance-recovered candidate table for later angular-momentum analysis.

## Inputs

- Candidate table: `data/processed/project3_orbital_diagnostics_candidates.csv`
- Recovery source selected by notebook: `{summary["recovery_source"]}`
- Merge key: `{summary["merge_key"]}`

## Outputs

- `notebooks/15_project3_distance_parallax_recovery.ipynb`
- `data/processed/project3_distance_recovered_candidates.csv`
- `data/processed/project3_distance_recovery_summary.csv`
- `report/project3_milestone4_distance_parallax_recovery.md`

## Recovery summary

- Number of Project 3 candidates: {summary["n_candidates"]}
- Candidates with recovered parallax: {summary["n_with_recovered_parallax"]}
- Candidates with positive parallax: {summary["n_with_positive_parallax"]}
- Candidates with inverse-parallax distance estimate: {summary["n_with_inverse_parallax_distance"]}
- Candidates passing basic angular-momentum readiness check: {summary["n_basic_angular_momentum_ready"]}

## Distance quality flags

{quality_lines}

## Notes and limitations

The recovered distance is currently an exploratory inverse-parallax estimate:

`distance_pc = 1000 / parallax_mas`

This is sufficient for Project 3 readiness and pipeline preparation, but it should not be treated as a final publication-grade distance model. Later orbit and angular-momentum analysis should document the adopted Galactic frame, solar parameters, parallax zero-point handling, and whether an external distance catalog or Bayesian distance estimate is used.

## Next step

Project 3 Milestone 5 can use `project3_distance_recovered_candidates.csv` to compute angular-momentum diagnostics such as `Lz`, `Lperp`, and `Ltot`, provided the recovered rows include sky position, proper motions, radial velocity, and a usable distance proxy.
'''

report_path.write_text(report_text, encoding="utf-8")
print("Wrote:", report_path)
print(report_text)

Wrote: /Users/liors/Documents/gaia-lamost-galactic-archaeology/report/project3_milestone4_distance_parallax_recovery.md
# Project 3 Milestone 4 — Distance / Parallax Recovery for Angular Momentum Analysis

## Objective

Project 3 Milestone 3 identified that the readiness-aware orbital diagnostics candidate table preserved useful velocity-space diagnostics but did not yet carry explicit parallax or distance information for the 27 Project 3 candidates.

This milestone checks whether the missing distance/parallax information can be recovered from existing processed Gaia–LAMOST larger feature or candidate tables, then writes a distance-recovered candidate table for later angular-momentum analysis.

## Inputs

- Candidate table: `data/processed/project3_orbital_diagnostics_candidates.csv`
- Recovery source selected by notebook: `gaia_lamost_larger_velocity_features.csv`
- Merge key: `source_id`

## Outputs

- `notebooks/15_project3_distance_parallax_recovery.ipynb`
- `data/processed/project